#### Training Module & Quiz Logic

In [4]:
import pandas as pd
import random
import json
from datetime import datetime

# Load data & engine functions
df_scenarios = pd.read_csv("warehouse_scenarios.csv")
df_checklist = pd.read_csv("compliance_checklist.csv")

# Paste or import the functions

def calculate_compliance_score(answers_dict):
    earned = 0.0
    possible = 0.0
    for _, row in df_checklist.iterrows():
        chk_id = row["check_id"]
        if chk_id not in answers_dict:
            continue
        answer = answers_dict[chk_id].strip().title()
        if answer == "N/A":
            continue
        possible += row["weight"]
        if answer == "Yes":
            earned += row["weight"]
        elif answer == "Partial":
            earned += row["weight"] * 0.5
    percentage = round((earned / possible) * 100, 1) if possible > 0 else 0.0
    return {
        "compliance_percentage": percentage,
        "points_earned": earned,
        "points_possible": possible
    }

def get_risk_category(compliance_pct):
    if compliance_pct >= 90: return "Low Risk", "green"
    if compliance_pct >= 70: return "Medium Risk", "orange"
    if compliance_pct >= 50: return "High Risk", "red"
    return "Critical Risk", "darkred"

def generate_findings_and_actions(answers_dict):
    findings = []
    for _, row in df_checklist.iterrows():
        chk_id = row["check_id"]
        if chk_id not in answers_dict: continue
        answer = answers_dict[chk_id].strip().title()
        if answer in ["No", "Partial"]:
            action = "Immediately address this issue. Review and update SOPs."
            if "expired" in row["requirement"].lower():
                action += " Move to quarantine and document loss."
            if "fifo" in row["requirement"].lower() or "fefo" in row["requirement"].lower():
                action += " Implement FIFO/FEFO system."
            if "ledger" in row["requirement"].lower() or "bin card" in row["requirement"].lower():
                action += " Conduct full stock count."
            findings.append({
                "check_id": chk_id,
                "requirement": row["requirement"],
                "answer": answer,
                "weight": row["weight"],
                "suggested_action": action
            })
    findings_sorted = sorted(findings, key=lambda x: (-x["weight"]))
    return findings_sorted

#### 1. Training / Quiz Flow Logic

For each selected scenario:
- Show scenario description first
- Ask questions one by one (random order or category-based)
- User selects: Yes / Partial / No / N/A
- Show brief explanation / feedback after each answer
- At the end: summary screen with score & recommendations

In [5]:
def run_quiz_simulation(scenario_id="SCN-001", shuffle_questions=True):
    """
    Simulates one full training/quiz session for a given scenario.
    Returns final result dictionary.
    """
    # Get scenario
    scenario = df_scenarios[df_scenarios["scenario_id"] == scenario_id].iloc[0]
    
    # Prepare questions (all checklist items)
    questions = df_checklist.copy()
    if shuffle_questions:
        questions = questions.sample(frac=1).reset_index(drop=True)
    
    # Store user answers
    user_answers = {}
    
    print(f"\n=== Training / Audit Simulation ===")
    print(f"Scenario: {scenario['title']} ({scenario['country']})")
    print(f"Description: {scenario['description']}")
    print(f"Key challenges: {', '.join(scenario['key_challenges'])}\n")
    
    # Simulate asking questions (in real app this will be interactive)
    for i, row in questions.iterrows():
        print(f"\nQuestion {i+1}/{len(questions)}")
        print(f"Category: {row['category']}")
        print(f"Requirement: {row['requirement']}")
        print(f"Evidence to check: {row['evidence']}")
        
        # Simulated user input (in notebook we hardcode or random for testing)
        # In real Streamlit: use st.radio or selectbox
        possible_answers = ["Yes", "Partial", "No", "N/A"]
        simulated_answer = random.choice(possible_answers)  # ← replace with real input later
        
        print(f"Selected answer: {simulated_answer}")
        user_answers[row["check_id"]] = simulated_answer
        
        # Immediate feedback
        if simulated_answer == "Yes":
            print("→ Correct – well done!")
        elif simulated_answer == "Partial":
            print("→ Partial compliance – room for improvement.")
        elif simulated_answer == "No":
            print("→ Non-compliant – critical issue detected.")
        else:
            print("→ N/A – skipped from scoring.")
    
    # Final scoring
    result = calculate_compliance_score(user_answers)
    risk_level, risk_color = get_risk_category(result["compliance_percentage"])
    
    findings = generate_findings_and_actions(user_answers)
    
    final_result = {
        "scenario_id": scenario_id,
        "scenario_title": scenario["title"],
        "compliance_percentage": result["compliance_percentage"],
        "risk_level": risk_level,
        "risk_color": risk_color,
        "points_earned": result["points_earned"],
        "points_possible": result["points_possible"],
        "findings": findings,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return final_result

#### 2. Run a Full Test Simulation

In [6]:
# Test with one scenario
test_result = run_quiz_simulation(scenario_id="SCN-001")

print("\n" + "="*60)
print("FINAL TRAINING / AUDIT RESULT")
print("="*60)
print(json.dumps(test_result, indent=2))

print("\nTop 3 Findings:")
for f in test_result["findings"][:3]:
    print(f"• {f['requirement']} ({f['answer']}, weight {f['weight']})")
    print(f"  → {f['suggested_action']}\n")


=== Training / Audit Simulation ===
Scenario: Flood-damaged warehouse in Juba (South Sudan)
Description: Heavy flooding has damaged roofing and floors. Some pallets of nutrition supplements are wet. Access roads are partially cut off.
Key challenges: [, ', D, a, m, a, g, e, d,  , s, t, o, c, k, ', ,,  , ', R, i, s, k,  , o, f,  , c, o, n, t, a, m, i, n, a, t, i, o, n, ', ,,  , ', D, e, l, a, y, e, d,  , d, i, s, t, r, i, b, u, t, i, o, n, ', ,,  , ', D, o, c, u, m, e, n, t, a, t, i, o, n,  , o, f,  , l, o, s, s, e, s, ', ]


Question 1/10
Category: Expiry Management
Requirement: Expiry dates tracked and visible
Evidence to check: Stock report showing expiry dates
Selected answer: N/A
→ N/A – skipped from scoring.

Question 2/10
Category: Documentation
Requirement: Updated stock ledger / bin cards for each item
Evidence to check: Sample bin cards / ledger entries
Selected answer: Partial
→ Partial compliance – room for improvement.

Question 3/10
Category: Documentation
Requirement: Wa